In [1]:
%pip install datasets math_verify vllm torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

In [ ]:
import gc
import torch
from vllm import LLM, SamplingParams
from math_verify import parse, verify

def clear_gpu_memory():
    for var in ["llm", "generations"]:
        if var in globals():
            del globals()[var]
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

clear_gpu_memory()

llm = LLM(model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
sampling_params = SamplingParams(n=8, max_tokens=2**10)

prompts = ["What is 13*17?"]
golds = ["221"]

generations = llm.generate(prompts, sampling_params)
for prompt, gold, generation in zip(prompts, golds, generations):
    gold = parse(gold)
    
    print("*"*50 + " Prompt " + "*"*50)
    print(prompt)
    
    for i, output in enumerate(generation.outputs):
        text = output.text
        answer = parse(text)
        correct = gold[0] == answer[0]
        
        print("*"*50 + f" Generation {i+1}: {answer[0]} ({"correct" if correct else "incorrect"}) ", "*"*50)
        if correct:
            print(text)

In [ ]:
from datasets import Dataset

def generate_dataset(llm, sampling_params, prompts, golds):
    dataset_dict = {
        "prompt": [],
        "outputs": [],
        "advantages": [],
    }
    generations = llm.generate(prompts, sampling_params)
    for prompt, gold, generation in zip(prompts, golds, generations):
        outputs = []
        rewards = []
        for output in generation.outputs:
            outputs.append(output.text)
            rewards.append(1.0 if verify(parse(gold), parse(output.text)) else 0.0)
        rewards = torch.tensor(rewards)
        
        std = rewards.std()
        if std == 0:
            advantages = torch.zeros_like(rewards)
        else:
            advantages = (rewards - rewards.mean()) / std
        
        dataset_dict["prompt"].append(prompt)
        dataset_dict["outputs"].append(outputs)
        dataset_dict["advantages"].append(advantages)
    return Dataset.from_dict(dataset_dict)

clear_gpu_memory()

llm = LLM(model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")
sampling_params = SamplingParams(n=64, max_tokens=2**15)

input_ds = (
    ds["train"]
    .shuffle()
    .select(range(0, 128))
)

output_ds = generate_dataset(
    llm=llm,
    sampling_params=sampling_params,
    prompts=input_ds["problem"],
    golds=input_ds["answers"]
)
output_ds.to_parquet("reg_grpo_128.parquet")
output_ds